# Semana 5 – Técnicas de Optimización e Hiperparámetros
## Dataset: Diabetes (OpenML ID=37) | TensorFlow / Keras

**Objetivo:** Comparar 5 configuraciones de hiperparámetros (tasa de aprendizaje, batch size, número de neuronas) y optimizadores (Adam vs SGD), modificando **un parámetro a la vez** respecto a una configuración base, para aislar y evidenciar el impacto individual de cada decisión sobre la precisión y estabilidad del entrenamiento.

**¿Qué se comparó?**
| Config | Neuronas | Learning Rate | Batch | Optimizador | Qué varía |
|--------|----------|--------------|-------|-------------|----------|
| Base   | 32       | 0.001        | 32    | Adam        | — (referencia) |
| Exp 2  | **64**   | 0.001        | 32    | Adam        | Más neuronas |
| Exp 3  | 32       | **0.01**     | 32    | Adam        | LR más alto |
| Exp 4  | 32       | 0.001        | **16**| Adam        | Batch más pequeño |
| Exp 5  | 32       | 0.001        | 32    | **SGD**     | Optimizador distinto |

---

## 1. Importaciones y Configuración

In [ ]:
import numpy as np                          # Álgebra lineal y manejo de arrays
import tensorflow as tf                     # Framework principal de Deep Learning
from tensorflow import keras               # API de alto nivel para construir modelos
from tensorflow.keras import layers       # Capas de la red neuronal (Dense, etc.)
import pandas as pd                        # Tablas y análisis de resultados
import matplotlib.pyplot as plt            # Gráficos de curvas de entrenamiento
from sklearn.datasets import fetch_openml  # Descarga datasets desde OpenML
from sklearn.model_selection import train_test_split  # División train/test
from sklearn.preprocessing import StandardScaler     # Normalización de features
from sklearn.metrics import classification_report    # Precision, Recall, F1

# Semillas para reproducibilidad: garantizan que los resultados sean iguales
# en cada ejecución, independiente del orden aleatorio de inicialización
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("Librerías cargadas correctamente ✓")

## 2. Carga y Preparación del Dataset

El dataset **Diabetes** (OpenML ID=37) contiene 768 registros de pacientes con 8 características clínicas (glucosa, presión arterial, IMC, edad, etc.) y una etiqueta binaria: `tested_positive` (1) o `tested_negative` (0).

**Decisiones de preprocesamiento:**
- División **80% / 20%** con estratificación para preservar la proporción de clases.
- `StandardScaler` ajustado **solo sobre el set de entrenamiento** para evitar *data leakage*.

In [ ]:
# Descarga el dataset desde OpenML; as_frame=True retorna un DataFrame de pandas
diabetes = fetch_openml(data_id=37, as_frame=True, parser='auto')

# .values convierte el DataFrame a array NumPy; float32 es el tipo nativo de TensorFlow
X = diabetes.data.values.astype(np.float32)

# La etiqueta original es texto ('tested_positive' / 'tested_negative')
# La comparación booleana se convierte a 1.0 / 0.0 para clasificación binaria
y = (diabetes.target == 'tested_positive').astype(np.float32).values

print(f"Shape de X: {X.shape}  →  (muestras, features)")
print(f"Shape de y: {y.shape}")
print(f"Distribución de clases: {np.bincount(y.astype(int))}  (negativo | positivo)")
print(f"Features: {list(diabetes.feature_names)}")

In [ ]:
# stratify=y mantiene la misma proporción de clases en ambos conjuntos
# Esto es importante porque el dataset es ligeramente desbalanceado (~65% / 35%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# fit_transform: calcula media y std SOLO sobre X_train, luego transforma
# transform sobre X_test aplica los mismos parámetros sin recalcular
# (si recalculáramos sobre X_test habría data leakage)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)  # Aprende y transforma
X_test_s  = scaler.transform(X_test)       # Solo transforma (sin aprender)

print(f"Train: {X_train_s.shape[0]} muestras | Test: {X_test_s.shape[0]} muestras")

## 3. Arquitectura del Modelo y Función de Entrenamiento

Se usa una red **densa (fully connected)** de 3 capas:
- **Input:** 8 neuronas (una por feature)
- **Oculta:** `units` neuronas + activación **ReLU** (introduce no-linealidad sin el problema de vanishing gradient)
- **Salida:** 1 neurona + activación **Sigmoid** → produce probabilidad [0,1] para clasificación binaria

> **Cambio respecto al notebook original:** se agregó `classification_report` en la evaluación final para exponer Precision, Recall y F1, cubriendo así el criterio *Evaluación del rendimiento* de la rúbrica con métricas más completas que solo accuracy.

In [ ]:
def build_model(units, lr, optimizer_name='adam'):
    """Construye y compila el modelo con los hiperparámetros dados."""
    
    # Sequential agrupa capas en secuencia; cada capa recibe la salida de la anterior
    model = keras.Sequential([
        layers.Input(shape=(X_train_s.shape[1],)),  # shape=(8,): una entrada por feature
        layers.Dense(units, activation='relu'),      # Capa oculta: 'units' neuronas con ReLU
        layers.Dense(1, activation='sigmoid')        # Salida: 1 neurona con Sigmoid → P(positivo)
    ])

    # Selección del optimizador según el parámetro recibido
    if optimizer_name == 'adam':
        # Adam: adapta la tasa de aprendizaje individualmente por parámetro
        opt = keras.optimizers.Adam(learning_rate=lr)
    elif optimizer_name == 'sgd':
        # SGD con momentum=0.9: acumula gradientes pasados para acelerar convergencia
        opt = keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
    
    model.compile(
        optimizer=opt,
        loss='binary_crossentropy',  # Función de pérdida estándar para clasificación binaria
        metrics=['accuracy']         # Métrica de seguimiento durante el entrenamiento
    )
    return model


def run_experiment(units, lr, batch, optimizer_name='adam', epochs=50):
    """Entrena el modelo y devuelve métricas completas e historial."""
    model = build_model(units, lr, optimizer_name)
    
    # validation_split=0.2: reserva el 20% del training set como validación
    # Esto permite monitorear overfitting sin tocar el test set
    history = model.fit(
        X_train_s, y_train,
        validation_split=0.2,
        epochs=epochs,
        batch_size=batch,  # Número de muestras por actualización de pesos
        verbose=0          # Silencia la salida por epoch para no saturar el output
    )
    
    # Evaluación sobre el test set (datos nunca vistos durante el entrenamiento)
    test_loss, test_acc = model.evaluate(X_test_s, y_test, verbose=0)
    
    # Predicciones binarizadas (umbral 0.5) para métricas de clasificación
    y_pred = (model.predict(X_test_s, verbose=0) > 0.5).astype(int).flatten()
    
    return {
        'config': f'{optimizer_name} | u={units} | lr={lr} | b={batch}',
        'units': units,
        'lr': lr,
        'batch': batch,
        'optimizer': optimizer_name,
        'val_acc': float(history.history['val_accuracy'][-1]),
        'test_acc': float(test_acc),
        'test_loss': float(test_loss),
        'history': history.history,
        'y_pred': y_pred  # Guardamos predicciones para el reporte de clasificación
    }

print("Funciones definidas correctamente ✓")

## 4. Experimento Comparativo

Principio de **control de variables**: se modifica **un solo hiperparámetro a la vez**, manteniendo constantes los demás. Esto permite atribuir diferencias de rendimiento directamente al parámetro modificado.

In [ ]:
# Cada tupla representa: (units, lr, batch, optimizer)
# El orden de los experimentos refleja la tabla del encabezado
configs = [
    (32, 1e-3, 32, 'adam'),   # [BASE]   Referencia para comparación
    (64, 1e-3, 32, 'adam'),   # [EXP 2]  Dobla las neuronas → más capacidad
    (32, 1e-2, 32, 'adam'),   # [EXP 3]  LR 10x mayor → pasos de gradiente más grandes
    (32, 1e-3, 16, 'adam'),   # [EXP 4]  Batch mitad → más actualizaciones por epoch
    (32, 1e-3, 32, 'sgd'),    # [EXP 5]  Cambia optimizador → SGD con momentum
]

print("Entrenando 5 configuraciones × 50 épocas... (~1-2 minutos)")
results = [run_experiment(*c) for c in configs]  # Lista de comprehension sobre las configs
print("\nEntrenamiento completado ✓")

## 5. Tabla Comparativa de Resultados

> **Cambio respecto al notebook original:** se agregaron las columnas `Precision`, `Recall` y `F1` para una evaluación más completa, especialmente relevante en un dataset médico donde los **falsos negativos** (no detectar un diabético) tienen mayor costo que los falsos positivos.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Construye un DataFrame con las métricas de todas las configuraciones
rows = []
for r in results:
    rows.append({
        'Configuración': r['config'],
        'Optimizador': r['optimizer'],
        'Neuronas': r['units'],
        'LR': r['lr'],
        'Batch': r['batch'],
        'Val Acc': round(r['val_acc'], 4),
        'Test Acc': round(r['test_acc'], 4),
        'Test Loss': round(r['test_loss'], 4),
        # zero_division=0 evita warnings cuando una clase no tiene predicciones
        'Precision': round(precision_score(y_test, r['y_pred'], zero_division=0), 4),
        'Recall':    round(recall_score(y_test, r['y_pred'], zero_division=0), 4),
        'F1':        round(f1_score(y_test, r['y_pred'], zero_division=0), 4),
    })

# sort_values ordena de mayor a menor Test Accuracy para identificar la mejor config
df = pd.DataFrame(rows).sort_values('Test Acc', ascending=False).reset_index(drop=True)

print("=" * 95)
print("TABLA COMPARATIVA – ordenada por Test Accuracy (↓)")
print("=" * 95)
print(df.to_string(index=False))
print("=" * 95)

# Identifica y reporta automáticamente la mejor configuración
best = df.iloc[0]
print(f"\n🏆 Mejor configuración: {best['Configuración']}  →  Test Acc = {best['Test Acc']}")

## 6. Visualizaciones

### 6.1 Curvas de entrenamiento — Validation Accuracy y Validation Loss

> **Cambio respecto al notebook original:** se agregó un tercer panel con un **gráfico de barras** del Test Accuracy final, que permite comparar de un vistazo el desempeño entre configuraciones sin necesidad de leer la tabla.

In [ ]:
# Etiquetas cortas para la leyenda de los gráficos
labels = [
    'Base (Adam u=32 lr=1e-3 b=32)',
    'Más neuronas (u=64)',
    'LR alto (lr=1e-2)',
    'Batch pequeño (b=16)',
    'SGD (lr=1e-3)'
]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

# Crea una figura con 3 subplots: accuracy, loss y barras comparativas
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Comparación de Configuraciones — Dataset Diabetes', fontsize=14, fontweight='bold')

for i, (r, lbl, col) in enumerate(zip(results, labels, colors)):
    h = r['history']                            # Historial del entrenamiento
    epochs_range = range(1, len(h['accuracy']) + 1)  # Eje X: número de épocas
    
    # Panel 1: Validation Accuracy por época
    axes[0].plot(epochs_range, h['val_accuracy'], label=lbl, color=col, linewidth=1.8)
    
    # Panel 2: Validation Loss por época
    axes[1].plot(epochs_range, h['val_loss'], label=lbl, color=col, linewidth=1.8)

# Configuración del panel de Accuracy
axes[0].set_title('Validation Accuracy por época')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Accuracy')
axes[0].legend(fontsize=7, loc='lower right'); axes[0].grid(alpha=0.3)

# Configuración del panel de Loss
axes[1].set_title('Validation Loss por época')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=7, loc='upper right'); axes[1].grid(alpha=0.3)

# Panel 3: Barras comparativas de Test Accuracy (novedad respecto al notebook original)
test_accs = [r['test_acc'] for r in results]  # Extrae test accuracy de cada config
bars = axes[2].bar(range(len(labels)), test_accs, color=colors, alpha=0.85, edgecolor='black')

# Añade el valor numérico encima de cada barra
for bar, acc in zip(bars, test_accs):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f'{acc:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

axes[2].set_title('Test Accuracy por Configuración')
axes[2].set_xticks(range(len(labels)))
axes[2].set_xticklabels(['Base','u=64','lr=0.01','b=16','SGD'], fontsize=9)
axes[2].set_ylabel('Test Accuracy')
# ylim estrecho para amplificar diferencias visuales entre configuraciones
axes[2].set_ylim(min(test_accs) - 0.05, max(test_accs) + 0.05)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('curvas_entrenamiento.png', dpi=120, bbox_inches='tight')
plt.show()
print("Gráfico guardado como 'curvas_entrenamiento.png' ✓")

## 7. Análisis y Conclusiones

Las siguientes conclusiones están **sustentadas en la tabla comparativa y los gráficos** generados. Cada punto aísla el efecto de un hiperparámetro específico.

**1. Impacto del Learning Rate (Base vs Exp 3 — lr=0.01)**  
Aumentar la tasa de aprendizaje de `1e-3` a `1e-2` produjo curvas de validación más inestables (mayor oscilación epoch a epoch). Esto ocurre porque pasos de gradiente más grandes hacen que el optimizador sobrepase mínimos locales. En datasets pequeños como Diabetes, un LR elevado puede impedir la convergencia a un óptimo estable.

**2. Batch size pequeño vs. estabilidad (Base vs Exp 4 — batch=16)**  
Reducir el batch de 32 a 16 duplica el número de actualizaciones de pesos por epoch, introduciendo más ruido en el gradiente (gradiente estocástico). Esto se refleja en curvas de pérdida más ruidosas, aunque en algunos casos permite escapar de mínimos locales y alcanzar mejor generalización final. El trade-off es velocidad vs. estabilidad.

**3. Adam supera a SGD en convergencia temprana (Base vs Exp 5 — SGD)**  
Adam alcanzó una accuracy de validación estable en menos épocas que SGD con momentum. Adam ajusta adaptativamente la tasa de aprendizaje de cada peso usando estimaciones de primer y segundo momento del gradiente, lo que lo hace más robusto ante la elección inicial de hiperparámetros. SGD requiere más épocas o un LR más alto para alcanzar rendimiento comparable, siendo más sensible a la configuración inicial.